# Часть 1. Проверка гипотезы в Python и составление аналитической записки

Вы предобработали данные в SQL, и теперь они готовы для проверки гипотезы в Python. Загрузите данные пользователей из Москвы и Санкт-Петербурга c суммой часов их активности из файла yandex_knigi_data.csv. Если работаете локально, скачать файл можно по ссылке.

Проверьте наличие дубликатов в идентификаторах пользователей. Сравните размеры групп, их статистики и распределение.

Напомним, как выглядит гипотеза: пользователи из Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении, чем пользователи из Москвы. Попробуйте статистически это доказать, используя одностороннюю проверку гипотезы с двумя выборками:

Нулевая гипотеза $H_0: \mu_{\text{СПб}} \leq \mu_{\text{Москва}}$ <br> Среднее время активности пользователей в Санкт-Петербурге не больше, чем в Москве.

Альтернативная гипотеза $H_1: \mu_{\text{СПб}} > \mu_{\text{Москва}}$ <br> Среднее время активности пользователей в Санкт-Петербурге больше, и это различие статистически значимо.

## Анализ времени взаимодействия с контентом приложения Яндекс.Книги пользователей из Москвы и Санкт-Петербурга

- Автор: Бусыгина Дарья
- Дата: 17.10.25

## Цели и задачи проекта

<font color='#777778'>Цель проекта состоит в проверке гипотезы, что пользователи из Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении, чем пользователи из Москвы. А также провести тестирование для улучшения сайта и упрощения интерфейса.</font>

## Описание данных

Таблица участников тестов.
Структура файла:
- user_id — идентификатор пользователя;
- group — группа пользователя;
- ab_test — название теста;
- device — устройство, с которого происходила регистрация.

Архив с одним csv-файлом, в котором собраны события 2020 года;
Структура файла:
- user_id — идентификатор пользователя;
- event_dt — дата и время события;
- event_name — тип события;
- details — дополнительные данные о событии.</font>

## Содержимое проекта

<font color='#777778'>Этот проект состоит из двух частей. В первой части завершим работу с данными Яндекс Книг. Во второй — разберём другие данные и проверите результаты A/B-тестирования</font>

---

## 1. Загрузка данных и знакомство с ними

Загрузите данные пользователей из Москвы и Санкт-Петербурга c их активностью (суммой часов чтения и прослушивания) из файла `/datasets/yandex_knigi_data.csv`.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats as st
from datetime import timedelta
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
from statsmodels.stats.proportion import proportions_ztest

In [2]:
active = pd.read_csv('/datasets/yandex_knigi_data.csv')

In [3]:
display(active.head())
display(active.info())

duplicates = active['puid'].duplicated().sum()
print('Кличество дубликатов:', duplicates)

,Unnamed: 0,city,puid,hours
0,0,Москва,9668,26.167776
1,1,Москва,16598,82.111217
2,2,Москва,80401,4.656906
3,3,Москва,140205,1.840556
4,4,Москва,248755,151.326434


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8784 entries, 0 to 8783
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  8784 non-null   int64  
 1   city        8784 non-null   object 
 2   puid        8784 non-null   int64  
 3   hours       8784 non-null   float64
dtypes: float64(1), int64(2), object(1)
memory usage: 274.6+ KB


None

Кличество дубликатов: 244


In [4]:
active = active.drop(columns=['Unnamed: 0'])

In [5]:
active = active.drop_duplicates(subset='puid')
print('Кличество строк после удаления:', len(active))

Кличество строк после удаления: 8540


В исходной таблице 4 столбца и 8784 строки, дубликатов идентификатора пользователя - 244, после их удаления строк осталось 8540

## 2. Проверка гипотезы в Python

Гипотеза звучит так: пользователи из Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении, чем пользователи из Москвы. Попробуйте статистически это доказать, используя одностороннюю проверку гипотезы с двумя выборками:

- Нулевая гипотеза H₀: Средняя активность пользователей в часах в двух группах (Москва и Санкт-Петербург) не различается.

- Альтернативная гипотеза H₁: Средняя активность пользователей в Санкт-Петербурге больше, и это различие статистически значимо.

In [6]:
# разделили данные по городам
moscow = active.query('city == "Москва"')['hours']
spb = active.query('city == "Санкт-Петербург"')['hours']

alpha = 0.05
mean = moscow.mean()

results = st.ttest_1samp(spb, mean, alternative = 'greater')

print('Средняя активность в Москве:',round(mean,2))
print('Средняя активность в СПБ:', round(spb.mean(),2))
print('t-статистика', round(results.statistic, 3))
print('p-value:', results.pvalue)

Средняя активность в Москве: 10.88
Средняя активность в СПБ: 11.26
t-статистика 0.462
p-value: 0.3220075909972008


In [7]:
if results.pvalue < alpha:
    print('Отвергаем нулевую гипотезу - пользователи из СПБ проводят больше времени')
else:
    print('Не удалось отвергнуть нулевую гипотезу - различия статистически незначимы')

Не удалось отвергнуть нулевую гипотезу - различия статистически незначимы


## 3. Аналитическая записка
По результатам анализа данных подготовьте аналитическую записку, в которой опишете:

- Выбранный тип t-теста и уровень статистической значимости.

- Результат теста, или p-value.

- Вывод на основе полученного p-value, то есть интерпретацию результатов.

- Одну или две возможные причины, объясняющие полученные результаты.



Тип теста: односторонний t-test, уровень значимости 0.05

Результаты: 
- Среднее время активности в Москве - 10.88 часов, в Петербурге - 11.26 часов
- t-статистика - 0.462
- p-value - 0.322

Вывод: различия статистически незначимы, время активности примерно одинаковое

- Пользовательское поведение в двух столицах схоже, это объясняется одинаковым уровнем цифровизации и демографическими характеристиками

----

# Часть 2. Анализ результатов A/B-тестирования

Теперь вам нужно проанализировать другие данные. Представьте, что к вам обратились представители интернет-магазина BitMotion Kit, в котором продаются геймифицированные товары для тех, кто ведёт здоровый образ жизни. У него есть своя целевая аудитория, даже появились хиты продаж: эспандер со счётчиком и напоминанием, так и подстольный велотренажёр с Bluetooth.

В будущем компания хочет расширить ассортимент товаров. Но перед этим нужно решить одну проблему. Интерфейс онлайн-магазина слишком сложен для пользователей — об этом говорят отзывы.

Чтобы привлечь новых клиентов и увеличить число продаж, владельцы магазина разработали новую версию сайта и протестировали его на части пользователей. По задумке, это решение доказуемо повысит количество пользователей, которые совершат покупку.

Ваша задача — провести оценку результатов A/B-теста. В вашем распоряжении:

* данные о действиях пользователей и распределении их на группы,

* техническое задание.

Оцените корректность проведения теста и проанализируйте его результаты.

## 1. Опишите цели исследования.



Исследовать, приведет ли новый улучшенный интерфейс сайта к большим покупкам пользователей

## 2. Загрузите данные, оцените их целостность.


In [8]:
participants = pd.read_csv('https://code.s3.yandex.net/datasets/ab_test_participants.csv')
events = pd.read_csv('https://code.s3.yandex.net/datasets/ab_test_events.zip',
                     parse_dates=['event_dt'], low_memory=False)

display(participants.head())
display(participants.info())
display(events.head())
display(events.info())

,user_id,group,ab_test,device
0,0002CE61FF2C4011,B,interface_eu_test,Mac
1,001064FEAAB631A1,B,recommender_system_test,Android
2,001064FEAAB631A1,A,interface_eu_test,Android
3,0010A1C096941592,A,recommender_system_test,Android
4,001E72F50D1C48FA,A,interface_eu_test,Mac


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14525 entries, 0 to 14524
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   user_id  14525 non-null  object
 1   group    14525 non-null  object
 2   ab_test  14525 non-null  object
 3   device   14525 non-null  object
dtypes: object(4)
memory usage: 454.0+ KB


None

,user_id,event_dt,event_name,details
0,GLOBAL,2020-12-01 00:00:00,End of Black Friday Ads Campaign,ZONE_CODE15
1,CCBE9E7E99F94A08,2020-12-01 00:00:11,registration,0.0
2,GLOBAL,2020-12-01 00:00:25,product_page,NaN
3,CCBE9E7E99F94A08,2020-12-01 00:00:33,login,NaN
4,CCBE9E7E99F94A08,2020-12-01 00:00:52,product_page,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 787286 entries, 0 to 787285
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   user_id     787286 non-null  object        
 1   event_dt    787286 non-null  datetime64[ns]
 2   event_name  787286 non-null  object        
 3   details     249022 non-null  object        
dtypes: datetime64[ns](1), object(3)
memory usage: 24.0+ MB


None

In [9]:
print('participants:')
print('Участников всего:', len(participants))
print('Уникальных айди юзеров в таблице:', participants['user_id'].nunique())
print('Пропуски в таблице:', participants.isna().sum())

print('events')
print('Событий всего:', len(events))
print('Уникальных айди юзеров в таблице:', events['user_id'].nunique())
print('Пропуски в таблице:', events.isna().sum())

participants:
Участников всего: 14525
Уникальных айди юзеров в таблице: 13638
Пропуски в таблице: user_id    0
group      0
ab_test    0
device     0
dtype: int64
events
Событий всего: 787286
Уникальных айди юзеров в таблице: 144184
Пропуски в таблице: user_id            0
event_dt           0
event_name         0
details       538264
dtype: int64


Всего строк 14525, уникальных пользователей - 13638, есть дубликаты 887, пропусков нет. Дубликаты удалаем
В таблице с событиями: всего строк 787286, уникальных - 144184 (много пользователей, которые не участвовали в тесте), 538264 пропуска в доп сведениях. Неоходимо отфильтровать пользователей из теста, чтобы остались только релевантные события, относящиеся к группам А и В

In [10]:
participants = participants.drop_duplicates(subset='user_id')
print('Уникальных пользователей после очистки:', participants['user_id'].nunique())

participants = participants.query('ab_test == "interface_eu_test"')
print('Участников в нужном тесте:', participants['user_id'].nunique())

events = events.query('user_id in @participants.user_id')
print('Событий после фильтрации:', len(events))
print('Уникальных пользователей в событиях после фильтрации:', events['user_id'].nunique())

Уникальных пользователей после очистки: 13638
Участников в нужном тесте: 10403
Событий после фильтрации: 76658
Уникальных пользователей в событиях после фильтрации: 10403


## 3. По таблице `ab_test_participants` оцените корректность проведения теста:

   3\.1 Выделите пользователей, участвующих в тесте, и проверьте:

   - соответствие требованиям технического задания,

   - равномерность распределения пользователей по группам теста,

   - отсутствие пересечений с конкурирующим тестом (нет пользователей, участвующих одновременно в двух тестовых группах).

In [11]:
print(participants['group'].value_counts())
print(participants['group'].value_counts(normalize=True))

B    5229
A    5174
Name: group, dtype: int64
B    0.502643
A    0.497357
Name: group, dtype: float64


In [12]:
cross_users = participants.groupby('user_id')['group'].nunique()
print('Пользователи в двух группах сразу:', (cross_users>1).sum())

Пользователи в двух группах сразу: 0


В тесте участвует 10403 пользователя. распределение по группам нормальное - 50.2% и 49.8%. Пересечений между группами нет, все пользователи принадлежат только одному тесту

- определите горизонт анализа: рассчитайте время (лайфтайм) совершения события пользователем после регистрации и оставьте только те события, которые были выполнены в течение первых семи дней с момента регистрации;

In [13]:
first_event = events.groupby('user_id')['event_dt'].min().reset_index()
first_event.columns=['user_id', 'registration_date']

events = events.merge(first_event, on ='user_id', how='left')

events['lifetime_days'] = (events['event_dt'] - events['registration_date']).dt.days

events_7days = events.query('lifetime_days<7')

print('Событий за первые 7 дней:', len(events_7days))
print('Пользователей за первые 7 дней:', events_7days["user_id"].nunique())

Событий за первые 7 дней: 66262
Пользователей за первые 7 дней: 10403


Оцените достаточность выборки для получения статистически значимых результатов A/B-теста. Заданные параметры:

- базовый показатель конверсии — 30%,

- мощность теста — 80%,

- достоверность теста — 95%.

In [14]:
p1 = 0.3
p2 = 0.33
alpha = 0.05
power = 0.8 

effect = proportion_effectsize(p1, p2)

analysis = NormalIndPower()
sample_size = analysis.solve_power(effect_size=effect, power=power, alpha=alpha, ratio=1)

print(f'Необходимое количество пользователей в каждой группе: {round(sample_size)}')

Необходимое количество пользователей в каждой группе: 3762


- рассчитайте для каждой группы количество посетителей, сделавших покупку, и общее количество посетителей.

In [15]:
purchases = events_7days.query('event_name == "purchase"')[['user_id']].drop_duplicates()

purchases = purchases.merge(participants[['user_id', 'group']], on='user_id', how='left')

summary = (
    purchases.groupby('group')['user_id'].nunique()
    .to_frame('buyers')
    .merge(
        participants.groupby('group')['user_id'].nunique().to_frame('total_users'),
        on='group'
    )
)

summary['conversion'] = summary['buyers'] / summary['total_users']
print(summary)

       buyers  total_users  conversion
group                                 
A        1427         5174    0.275802
B        1532         5229    0.292981


- сделайте предварительный общий вывод об изменении пользовательской активности в тестовой группе по сравнению с контрольной.

В тестовой группе В наблюдается небольшое увеличение конверсии, на этом этапе различия умеренные, но потенциально положительные

## 4. Проведите оценку результатов A/B-тестирования:

- Проверьте изменение конверсии подходящим статистическим тестом, учитывая все этапы проверки гипотез.

- H0: Конверсии в группах равны - новый интерфейс не влияет на вероятность покупки
- H1: Конверсия в группе В выше, чем в группе А

In [16]:
successes = np.array(summary['buyers'])
samples = np.array(summary['total_users'])

z_stat, p_value = proportions_ztest(count=successes, nobs=samples, alternative = 'smaller')

print(f'Z-статистика: {z_stat:.4f}')
print(f'p-value: {p_value:.4f}')

Z-статистика: -1.9419
p-value: 0.0261


- Опишите выводы по проведённой оценке результатов A/B-тестирования. Что можно сказать про результаты A/B-тестирования? Был ли достигнут ожидаемый эффект в изменении конверсии?

Проведенный тест пропорций проверял гипотезу о различии конверсии между контрольной и тестовой группами

Результаты показали: p-value = 0.0261 < 0.05, следовательно, нулевая гипотез отвергается, значит, новая версия интерфейса действительно увеличит продажи

Однако по тз увеличение конферсии должно быть не менее 3pp, а фактический результат:
 - конверсия группы А - 27.6%
 - конверсия группы В - 29.3%
 - прирост +1.7pp
 
Таким образом, хоть тест и показал статистически значимое улучшение, ожидаемый эффект не достигнут, фактический прирост ниже 

Рекомендации:
 - не внедрять новую версию интерфейса для всех пользователей сразу, сначала усилить сайт и добавить новые элементы, затем провести повторное исследование
 - провети доп исследования: анализ воронки, кликов, времени на странице